# 04 평가: (i) 학생 원본 4bit, (ii) LoRA 학생 merge 후 4bit (배포본). (iii) 선생은 01에서 생성

| 항목 | 값 |
|---|---|
| GPU | A100 40GB 또는 80GB |
| 예상 시간 | 약 1.5~2시간 (모델마다 로드 5분, batch 1 시간 측정 50개 15분, 평가 934개 batch 16 생성 25~40분) |
| 예상 컴퓨트 | 약 20~25 CU (A100 약 13 CU/시간 가정. Colab 리소스 창에서 실제 소모율을 확인해 기록) |
| 디스크 | 학생 원본 59GB(캐시) + Drive NF4 9.6GB |
| 이어 하기 | 생성은 batch마다 Drive `eval/*.jsonl`에 이어 쓰고, 시간 측정은 json이 있으면 건너뛴다. |

사전 등록: `docs/spec.md` §6.1~6.3, `docs/decisions.md` 2026-11-10. 설정을 바꾸면 사전 등록 위반이니 멈추고 기록한다.
준비: repo에서 `scripts/pack_distill_bundle.sh`로 만든 `tomato_distill_bundle.tar.gz`를 Drive `MyDrive/tomato-distill/`에 올린다.

In [ ]:
# 공통 준비: Drive 연결, 묶음 풀기, 공용 코드 import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, time, gc
DIR = "/content/drive/MyDrive/tomato-distill"
os.environ["TOMATO_DISTILL_DIR"] = DIR
if not os.path.exists(f"{DIR}/bundle/notebooks/distill_common.py"):
    os.makedirs(f"{DIR}/bundle", exist_ok=True)
    !tar xzf {DIR}/tomato_distill_bundle.tar.gz -C {DIR}/bundle
sys.path.insert(0, f"{DIR}/bundle/notebooks")
import distill_common as dc
prompt = dc.setup()
from tomato.explain import grounding
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q -U "transformers>=5.9.0" peft bitsandbytes accelerate
import torch
evals = dc.eval_inputs()
timing = dc.timing_subset(evals)
E = dc.DRIVE / "eval"
print(len(evals), len(timing))

## 배포 형식 그대로 로드 (NF4, embedding만 CPU), greedy, 사고 출력 끔 (skip_reasoning)
계산 dtype은 fp16 하나로 (i)·(ii)를 평가한다(spec §6.2, 2026-11-12 확정. 데모 서버 RTX 2080 Ti는 bf16 미지원).

In [ ]:
COMPUTE_DTYPE = "fp16"  # spec §6.2 (2026-11-12 확정)
assert COMPUTE_DTYPE == dc.EVAL_COMPUTE_DTYPE
MODELS = [("student_base", dc.STUDENT_ID), ("student_lora", str(dc.DRIVE / "merged_nf4"))]
for label, path in MODELS:
    tpath, gpath = E / f"{label}_timing.json", E / f"{label}.jsonl"
    if tpath.exists() and len(dc.read_jsonl(gpath)) >= len(evals):
        continue
    model, tok = dc.load_student(path, compute=COMPUTE_DTYPE)
    print(label, "device map 확인:", {k: v for k, v in model.hf_device_map.items() if v == "cpu"})
    if not tpath.exists():
        t = dc.measure_latency(model, tok, prompt, timing)
        t.update({"model": label, "path": path, "prompt_version": prompt.PROMPT_VERSION, "compute_dtype": COMPUTE_DTYPE})
        tpath.write_text(json.dumps(t, ensure_ascii=False, indent=1))
        print(label, t["mean_sec"], "초", t["max_reserved_gb"], "GB")
    dc.student_generate(model, tok, prompt, evals, batch_size=16, out_path=gpath, label=label)
    del model; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 빠른 확인용 통과율. 최종 지표는 repo의 scripts/score_distill.py로 계산한다.
inputs = {r["id"]: r["input"] for r in evals}
for label in ("student_base", "student_lora", "teacher"):
    rows = dc.read_jsonl(E / f"{label}.jsonl")
    if rows:
        cs = [grounding.check(r["text"], inputs[r["id"]]) for r in rows]
        print(label, len(rows), "pass", round(sum(c["pass"] for c in cs) / len(cs), 3))

## repo로 가져올 파일
- `eval/student_base.jsonl`, `eval/student_lora.jsonl`, `eval/*_timing.json` → `distill/eval/`
- `merged_nf4/`를 데모 서버로 옮겨 `scripts/demo_gpu_check.py --timing`으로 VRAM을 잰다(채택 기준 2, spec §6.3).
- 그다음 repo에서 `uv run python scripts/score_distill.py`를 실행한다.